## Análisis y normalización de la duración de los audios

El objetivo de este notebook es:
1. Leer todos los audios organizados en la carpeta `data/audios_ordenados`.
2. Eliminar el **silencio inicial** (y final) para que la señal comience exactamente cuando empieza la vocalización de los pacientes.
3. Analizar la distribución estadística de las duraciones una vez eliminado el silencio muerto.
4. Encontrar y fijar un umbral $X$ en segundos para recortar todos los audios a la misma longitud, y de la misma parte inicial de la voz.
5. Guardar el nuevo set de datos ajustado temporalmente en una nueva carpeta.

In [ ]:
# Comenta esta línea si ya tienes instaladas las librerías:
# !pip install librosa tqdm matplotlib pandas soundfile

import os
import librosa
import numpy as np
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# Definición de Rutas principales
BASE_DIR = Path(os.getcwd()).parent.parent
INPUT_DIR = BASE_DIR / 'data' / 'audios_ordenados'
OUTPUT_DIR = BASE_DIR / 'data' / 'audios_longitud_fija'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
#### 1. LEER AUDIOS Y ELIMINAR SILENCIOS ####

all_wavs = list(INPUT_DIR.rglob('*.wav'))
print(f"Se encontraron {len(all_wavs)} audios en {INPUT_DIR}\n")

audio_data = []
# TOP_DB define los decibelios por debajo de los cuales se considera silencio puro. 
# 25-30 dB es un estándar para speech, se puede ajustar en caso de que borre parte de la voz.
TOP_DB = 25  

for wav_path in tqdm(all_wavs, desc="Quitando silencios"):
    try:
        # Cargar manteniendo el sampling rate nativo del archivo (sr=None)
        y, sr = librosa.load(wav_path, sr=None)
        
        # Eliminar silencios iniciales y finales (trim)
        y_trimmed, index = librosa.effects.trim(y, top_db=TOP_DB)
        
        # Obtener duración activa de la voz
        duration_trimmed = librosa.get_duration(y=y_trimmed, sr=sr)
        
        audio_data.append({
            'filepath': wav_path,
            'rel_path': wav_path.relative_to(INPUT_DIR),
            'sr': sr,
            'duration_trimmed': duration_trimmed,
            'y_trimmed': y_trimmed
        })
    except Exception as e:
        print(f"Error verificando {wav_path}: {e}")

df_durations = pd.DataFrame(audio_data)

In [ ]:
#### 2. ANÁLISIS DE LAS CONSTANTES DE TIEMPO RESULTANTES ####

plt.figure(figsize=(10, 5))
plt.hist(df_durations['duration_trimmed'], bins=40, color='mediumpurple', edgecolor='white')
plt.title('Distribución de duraciones REALES de la voz (sin silencio muertos)')
plt.xlabel('Duración (segundos)')
plt.ylabel('Frecuencia (cantidad de audios)')

# Líneas para métricas clave 
dur_min = df_durations['duration_trimmed'].min()
dur_med = df_durations['duration_trimmed'].median()
plt.axvline(dur_min, color='red', linestyle='dashed', linewidth=2, label=f"Mínimo: {dur_min:.3f}s")
plt.axvline(dur_med, color='green', linestyle='dashed', linewidth=2, label=f"Mediana: {dur_med:.3f}s")

plt.legend()
plt.grid(alpha=0.3)
plt.show()

print("Resumen estadístico de la duración en segundos (sin silencios):\n")
print(df_durations['duration_trimmed'].describe())

### 3. Normalizar todos los audios a la longitud objetivo 
Elige un umbral objetivo `TARGET_DURATION`. Si quieres no perder absolutamente nada y que todos los modelos coincidan exactamente, ponlo al mínimo del `describe()` anterior. Si prefieres un valor redondo como **1.0 segundos** y hay audios más pequeños que ese valor (por poco), el script aplicará ceros (padding) a rellenar el final. Todo lo que sea más largo, se cortará conservando exactamente tu rango de tiempo por igual para todos los pacientes.

In [ ]:
#### 3. CORTAR Y/O PADEAR CON PRECISIÓN Y GUARDAR  ####

##############################################################
# ⚠️ CONFIGURACIÓN: AJUSTA ESTE VALOR SEGÚN EL HISTOGRAMA
# Por defecto tomamos el valor mínimo estricto, o algo estándar (ej: 0.5 o 1.0).
TARGET_DURATION = df_durations['duration_trimmed'].min()  # Todos miden lo idéntico al más corto.
##############################################################

print(f"Se están transformando los audios para que duren exactamente {TARGET_DURATION:.4f} segundos.\n")

count_saved = 0
for row in tqdm(audio_data, desc="Procesando tamaño estático"):
    y = row['y_trimmed']
    sr = row['sr']
    out_rel_path = row['rel_path']
    out_abs_path = OUTPUT_DIR / out_rel_path
    
    # Crea carpeta de clase (Ej. HC/A) si no existe.
    out_abs_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Calcula número exacto de frames para X segundos según frecuencias (sr).
    target_samples = int(TARGET_DURATION * sr)
    
    if len(y) > target_samples:
        # Cortar los primeros X segundos del inicio real de la voz
        y_final = y[:target_samples]
    else:
        # Si por algo faltó 0.001 seg, rellenar el final con silencio para empatar
        padding_amount = target_samples - len(y)
        y_final = np.pad(y, (0, padding_amount), mode='constant')
        
    # Escribir el nuevo audio perfecto.
    sf.write(out_abs_path, y_final, sr)
    count_saved += 1

print(f"\n¡Completado! {count_saved} audios finalizados en longitud fija.\nConsulta en: {OUTPUT_DIR}")